# 06 - StreetCLIP fine-tuning: experiments (Faris)

Model **7** of the results table. Thea's notebook 03 used StreetCLIP **frozen** (zero-shot, then a linear probe on
fixed embeddings). Here we let the last part of the network itself change, on our images.

This notebook does **not train anything**. Trainings run on the IE SLURM cluster (`docs/cluster.md`). It
1. fixes the question and the decision rule **before** seeing any result,
2. defines the small experiment plan and prints the commands to launch it,
3. after the runs, reads `results/<run>_history.csv` and picks the best run **on val**.

It runs today with zero finished runs: the analysis cells then just say so.

In [1]:
import json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from geosense import compare, load_manifest
from geosense.data import PROJECT_ROOT

# --- configuration ---------------------------------------------------------------------------
RESULTS_DIR = PROJECT_ROOT / "results"     # where geosense.finetune writes <run>_history.csv / _config.json
FIG_DIR = PROJECT_ROOT / "results"         # figures of this notebook: ft_*.png
DATA_ROOT = "$DATA_ROOT"                    # compressed_dataset folder ON THE MACHINE THAT TRAINS (cluster: see docs/cluster.md)
N_BOOT = 1000                               # bootstrap resamples for the noise estimate
# ---------------------------------------------------------------------------------------------
# Reference scores from Thea's frozen models (real files, val split only)
y_val = load_manifest(split="val").label.to_numpy()
ref = {m: compare.macro_f1(y_val, np.load(PROJECT_ROOT / "predictions" / f"{m}_val_logits.npy").argmax(1))
       for m in ("streetclip_zeroshot", "streetclip_probe")}
print({m: round(v, 3) for m, v in ref.items()}, "| val images:", len(y_val))

{'streetclip_zeroshot': 0.695, 'streetclip_probe': 0.776} | val images: 7171


## 0. The question and the decision rule (fixed in advance)

**Question:** does fine-tuning part of StreetCLIP beat the linear probe, which only re-learns the last layer?

**Rules**, written before any run so that we cannot bend them after seeing the numbers:
- Every choice (method, number of blocks, learning rate, loss weighting, best epoch) is made on **val**. Test is never looked at here.
- **Primary metric: val macro-F1.** It gives every country the same weight, so small countries (Africa, ~15 val images
  for some) matter as much as the USA. Top-1 is reported but does not decide.
- **Fine-tuned becomes the final StreetCLIP model only if it beats the linear probe on val by more than the bootstrap
  noise**: the 95% interval of the paired bootstrap difference (fine-tuned − probe, notebook 07) must be entirely above 0.
  Otherwise we keep the probe: same accuracy for a much cheaper model.

Picking the best of ~8 runs on val is itself a small optimistic bias; this is why the final number is the test score,
measured once at the end.

## 1. Experiment plan

Small on purpose (8 runs, not a grid): each run answers **one** question, the cheapest and safest first.
Run 1 is a sanity check; if it fails, nothing else is launched. After runs 2–4, set `BEST_BLOCKS` to the winner;
after runs 5–6, set `BEST_LR`. All other options keep the defaults of `geosense.finetune`
(sqrt class weights, 8 epochs max, early stopping with patience 2, warm-up + cosine).

In [2]:
BEST_BLOCKS = 2      # update after runs 2-4 (the number of blocks with the best val macro-F1)
BEST_LR = "1e-5"     # update after runs 5-6

PLAN = [
    {"run": "ft_head", "args": "--method head",
     "why": "Sanity check: only the head is trained, so it should land near Thea's probe.",
     "changes_mind": "Far below the probe -> a bug in our pipeline (view, crop, labels): fix before spending GPU hours."},
    {"run": "ft_lb1", "args": "--method last_blocks --unfreeze-blocks 1 --lr 1e-5",
     "why": "Smallest real fine-tuning: the last transformer block adapts to our images.",
     "changes_mind": "Already clearly above the probe -> fine-tuning is worth it; if not, more blocks must show it."},
    {"run": "ft_lb2", "args": "--method last_blocks --unfreeze-blocks 2 --lr 1e-5",
     "why": "Default of the script: more capacity than 1 block, still cheap.",
     "changes_mind": "Better than 1 block -> capacity helps, try 4; equal -> stay small."},
    {"run": "ft_lb4", "args": "--method last_blocks --unfreeze-blocks 4 --lr 1e-5",
     "why": "Most capacity we try; where over-fitting (train loss down, val loss up) should show first.",
     "changes_mind": "Worse than 2 blocks -> we over-fit with 33k images; better -> capacity still limits us."},
    {"run": f"ft_lb{BEST_BLOCKS}_lr3e-6", "args": f"--method last_blocks --unfreeze-blocks {BEST_BLOCKS} --lr 3e-6",
     "why": "LR check below the default, on the best number of blocks.",
     "changes_mind": "Better than 1e-5 -> 1e-5 was damaging pre-trained features (forgetting)."},
    {"run": f"ft_lb{BEST_BLOCKS}_lr3e-5", "args": f"--method last_blocks --unfreeze-blocks {BEST_BLOCKS} --lr 3e-5",
     "why": "LR check above the default.",
     "changes_mind": "Better than 1e-5 -> we were too careful; a much worse or unstable curve confirms a small LR."},
    {"run": "ft_lora8", "args": "--method lora --lora-r 8 --lr 1e-4",
     "why": "LoRA: small adapters in ALL blocks instead of full weights in the last ones (LoRA needs a larger LR).",
     "changes_mind": "Matches last_blocks with fewer trainable parameters -> prefer LoRA (smaller checkpoint for the app)."},
    {"run": f"ft_lb{BEST_BLOCKS}_noweights",
     "args": f"--method last_blocks --unfreeze-blocks {BEST_BLOCKS} --lr {BEST_LR} --imbalance none",
     "why": "Imbalance check on the best config: plain cross-entropy instead of sqrt class weights.",
     "changes_mind": "Higher macro-F1 without weights -> the weights hurt; lower (expected) -> keep sqrt weights."},
]
pd.set_option("display.max_colwidth", None)
pd.DataFrame(PLAN).set_index("run")[["args", "why", "changes_mind"]]

,args,why,changes_mind
run,,,
ft_head,--method head,"Sanity check: only the head is trained, so it should land near Thea's probe.","Far below the probe -> a bug in our pipeline (view, crop, labels): fix before spending GPU hours."
ft_lb1,--method last_blocks --unfreeze-blocks 1 --lr 1e-5,Smallest real fine-tuning: the last transformer block adapts to our images.,"Already clearly above the probe -> fine-tuning is worth it; if not, more blocks must show it."
ft_lb2,--method last_blocks --unfreeze-blocks 2 --lr 1e-5,"Default of the script: more capacity than 1 block, still cheap.","Better than 1 block -> capacity helps, try 4; equal -> stay small."
ft_lb4,--method last_blocks --unfreeze-blocks 4 --lr 1e-5,"Most capacity we try; where over-fitting (train loss down, val loss up) should show first.",Worse than 2 blocks -> we over-fit with 33k images; better -> capacity still limits us.
ft_lb2_lr3e-6,--method last_blocks --unfreeze-blocks 2 --lr 3e-6,"LR check below the default, on the best number of blocks.",Better than 1e-5 -> 1e-5 was damaging pre-trained features (forgetting).
ft_lb2_lr3e-5,--method last_blocks --unfreeze-blocks 2 --lr 3e-5,LR check above the default.,Better than 1e-5 -> we were too careful; a much worse or unstable curve confirms a small LR.
ft_lora8,--method lora --lora-r 8 --lr 1e-4,LoRA: small adapters in ALL blocks instead of full weights in the last ones (LoRA needs a larger LR).,Matches last_blocks with fewer trainable parameters -> prefer LoRA (smaller checkpoint for the app).
ft_lb2_noweights,--method last_blocks --unfreeze-blocks 2 --lr 1e-5 --imbalance none,Imbalance check on the best config: plain cross-entropy instead of sqrt class weights.,Higher macro-F1 without weights -> the weights hurt; lower (expected) -> keep sqrt weights.


## 2. Commands (printed only, nothing is submitted)

Cluster: `scripts/slurm/submit.sh <run_name> <args>` submits one SLURM job (see `docs/cluster.md`). The direct command
is the same training without SLURM (e.g. on a GPU machine, or locally with `--subset 560 --epochs 1` as a quick
debugging run: subset runs are flagged and ignored by the analysis below). A job that stops can be continued with
the same command plus `--resume`.

In [3]:
for p in PLAN:
    print(f"# {p['run']}: {p['why']}")
    print(f"scripts/slurm/submit.sh {p['run']} {p['args']}")
    print(f"python -m geosense.finetune --root \"{DATA_ROOT}\" --run-name {p['run']} {p['args']}\n")

# ft_head: Sanity check: only the head is trained, so it should land near Thea's probe.
scripts/slurm/submit.sh ft_head --method head
python -m geosense.finetune --root "$DATA_ROOT" --run-name ft_head --method head

# ft_lb1: Smallest real fine-tuning: the last transformer block adapts to our images.
scripts/slurm/submit.sh ft_lb1 --method last_blocks --unfreeze-blocks 1 --lr 1e-5
python -m geosense.finetune --root "$DATA_ROOT" --run-name ft_lb1 --method last_blocks --unfreeze-blocks 1 --lr 1e-5

# ft_lb2: Default of the script: more capacity than 1 block, still cheap.
scripts/slurm/submit.sh ft_lb2 --method last_blocks --unfreeze-blocks 2 --lr 1e-5
python -m geosense.finetune --root "$DATA_ROOT" --run-name ft_lb2 --method last_blocks --unfreeze-blocks 2 --lr 1e-5

# ft_lb4: Most capacity we try; where over-fitting (train loss down, val loss up) should show first.
scripts/slurm/submit.sh ft_lb4 --method last_blocks --unfreeze-blocks 4 --lr 1e-5
python -m geosense.finetune --root "$DATA

## 3. After the runs: learning curves

Each training writes one row per epoch. **Epoch 0 is measured before any training** (the model starts exactly as
zero-shot), so every curve starts at the zero-shot score: a good run climbs from there quickly.
Dashed / dotted lines: Thea's zero-shot and linear probe, computed with the same val images.

In [4]:
hist = compare.load_run_histories(RESULTS_DIR)
debug = sorted(hist.loc[hist.subset > 0, "run"].unique()) if len(hist) else []
hist = hist[hist.subset == 0] if len(hist) else hist
if debug:
    print("debug (subset) runs, ignored:", debug)
if hist.empty:
    print(f"No finished runs yet in {RESULTS_DIR} (looking for <run>_history.csv). "
          "Launch the plan above, copy the results/ files back, then re-run this notebook.")
else:
    runs = list(hist.run.unique())
    colour = {r: plt.cm.tab10(i % 10) for i, r in enumerate(runs)}
    marker = {"head": "s", "last_blocks": "o", "lora": "^"}
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
    for r, h in hist.groupby("run", sort=False):
        kw = dict(color=colour[r], marker=marker.get(h.method.iloc[0], "x"), label=r)
        ax[0].plot(h.epoch, h.val_macro_f1, **kw)
        ax[0].annotate(r, (h.epoch.iloc[-1], h.val_macro_f1.iloc[-1]), fontsize=8, xytext=(4, 0), textcoords="offset points")
        ax[1].plot(h.epoch[h.epoch >= 1], h.train_loss[h.epoch >= 1], **kw)
        ax[2].plot(h.epoch, h.val_loss, **kw)
    ax[0].axhline(ref["streetclip_zeroshot"], ls="--", c="grey", label="zero-shot (Thea)")
    ax[0].axhline(ref["streetclip_probe"], ls=":", c="black", label="linear probe (Thea)")
    for a, t in zip(ax, ("val macro-F1", "train loss", "val loss")):
        a.set(xlabel="epoch (0 = before training)", ylabel=t, title=t)
        a.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax[0].legend(fontsize=8)
    fig.tight_layout(); fig.savefig(FIG_DIR / "ft_runs_curves.png", dpi=150); plt.show()

debug (subset) runs, ignored: ['localft_subset100']
No finished runs yet in <repo>/results (looking for <run>_history.csv). Launch the plan above, copy the results/ files back, then re-run this notebook.


### Results table (val)
`best_epoch` = epoch with the highest val macro-F1, i.e. the one saved in `best.pt` (0 = training never beat
zero-shot: that run is useless). `zero_shot_macro_f1`
= the epoch-0 row, which should be the same for every run (same model, same images): if not, a run used other settings.
`val_loss_up` = val loss at the last epoch minus its minimum: positive means the model got worse at the end.

In [5]:
if hist.empty:
    print("No runs yet: the table will appear here.")
else:
    table = compare.run_table(hist)
    table["val_loss_up"] = hist.groupby("run").val_loss.agg(lambda v: v.iloc[-1] - v.min())
    table["vs_probe"] = table.best_val_macro_f1 - ref["streetclip_probe"]
    table["gpu_hours"] = table.sec_per_epoch * table.epochs_done / 3600
    display(table.drop(columns="subset").sort_values("best_val_macro_f1", ascending=False).round(4))

No runs yet: the table will appear here.


### How to read the curves (over-fitting diagnosis)
- **Healthy:** val macro-F1 rises in the first 1–3 epochs then flattens; train and val loss both go down.
- **Over-fitting:** train loss keeps falling while val loss turns up (`val_loss_up` > 0) and val macro-F1 stalls or
  drops. Early stopping then keeps the best epoch, but more blocks / a higher LR would make it worse: prefer the
  smaller run.
- **Too high LR / forgetting:** val macro-F1 *falls below epoch 0* in the first epoch (the pre-trained features are
  damaged faster than the head learns). Look at the `lr3e-5` run.
- **Too low LR:** the curve barely leaves the zero-shot line and is still rising at the last epoch.
- **Sanity run (`ft_head`):** should end close to the dotted probe line. Small differences are normal (augmentation,
  other optimiser), a large gap is a bug.
- Differences of ~0.01 macro-F1 between runs are likely noise (see the bootstrap in notebook 07): do not over-read them.

## 4. Choose the best run (val) and finish the job
Best run = highest val macro-F1 at its best epoch, debug runs excluded. Then two exports, in this order. Both read
`best.pt` of that run and rebuild the model from the config saved inside it, so no training options are needed.
1. **val** with `--final` → `predictions/streetclip_finetuned_val_logits.npy`, read by notebook 07 for the verdict;
2. **test**, **ONCE, at the very end, after the team agrees** (notebook 07 verdict + Maya's evaluation ready).

On the cluster, `scripts/slurm/submit.sh --export <run>` (without `--final`) only writes
`predictions/runs/<run>_val_logits.npy`: notebook 07 shows it as `run:<run>` but the verdict needs the contract file.

In [6]:
best = compare.pick_best_run(hist) if not hist.empty else None
if best is None:
    print("No finished (non-subset) run yet: nothing to choose.")
else:
    row = compare.run_table(hist).loc[best]
    print(f"Best run on val: {best}  (val macro-F1 {row.best_val_macro_f1:.4f} at epoch {row.best_epoch}, "
          f"probe {ref['streetclip_probe']:.4f})\n")
    print("# 1) export VAL of the chosen run as the contract file (for notebook 07)")
    print(f"scripts/slurm/submit.sh --export {best} --final")
    print(f"python -m geosense.finetune --root \"{DATA_ROOT}\" --run-name {best} --export-only --export val --final\n")
    print("# " + "=" * 90)
    print("# 2) ONCE, AT THE VERY END, AFTER THE TEAM AGREES - TEST export. Never use test to choose anything.")
    print(f"#    FINAL_TEST=1 scripts/slurm/submit.sh --export {best}")
    print(f"#    python -m geosense.finetune --root \"{DATA_ROOT}\" --run-name {best} --export-only --export val,test --final")
    print("# " + "=" * 90)

No finished (non-subset) run yet: nothing to choose.


## 5. Notes for the report
- **What fine-tuning changes vs the probe.** The probe keeps every image embedding fixed and only moves the 56 class
  vectors. Fine-tuning also changes how the image is turned into an embedding (last blocks, or LoRA adapters), so the
  model can pay more attention to cues that matter for *our* 56 countries (road markings, signs, poles, vegetation).
- **Why a small learning rate (1e-5, 100× smaller than usual).** The weights are already very good; a large step
  would overwrite what StreetCLIP learnt from 1.1M images in a few hundred steps. The head gets a larger LR (1e-4)
  because it is the part that must move most.
- **Why start from zero-shot.** The head is initialised with the text embeddings of the country prompts, so epoch 0
  *is* the zero-shot model: training can only start from a good point, and the first gradient steps are not wasted
  on a random head that would send large, noisy gradients into the backbone.
- **Catastrophic forgetting.** Fitting 33k images can erase general knowledge from pre-training. Small LR, few
  unfrozen blocks / LoRA, few epochs and early stopping on val all limit it; the `lr3e-5` and `lb4` runs show the risk.
- **Cost.** Report GPU hours per run (`gpu_hours` column) and in total: the probe needs one pass over the images,
  fine-tuning needs one forward + backward pass per epoch. If fine-tuning only matches the probe, the probe wins on cost.
- **Over-confidence.** Fine-tuned models are often too sure of themselves (see NLL / confidence in notebook 07):
  Maya's temperature scaling on val fixes the probabilities without changing the predictions.